## Environment Setup

This notebook runs on a dedicated conda environment, **not** the notebook server's default kernel — select **"VBLL Repro (py3.12)"** from the kernel picker before running any cells below.

### Location & Instructions for those setting up in GPVMs 
```
/exp/icarus/data/users/sdey2/vbll_surrogate/conda/envs/vbll_repro
```
Deliberately **not** under `nashome` (home directory) — that quota is nearly full and will fail on package installs, kernel registration, or anything else that writes files. Always use the `/exp/icarus/data/` path for anything sizeable.

### Package versions
| Package | Version |
|---|---|
| Python | 3.12.12 |
| torch | 2.6.0+cpu |
| vbll | 0.4.9 |
| numpy | 2.4.4 |
| pandas | 3.0.2 |

### One-time setup (already done for this env — reference only)

```bash
# Create the env at the correct location
conda create -p /exp/icarus/data/users/sdey2/vbll_surrogate/conda/envs/vbll_repro python=3.12 -y

# Set a shell variable for convenience (add to ~/.bashrc to persist across sessions)
ENV=/exp/icarus/data/users/sdey2/vbll_surrogate/conda/envs/vbll_repro

# Install core packages — CPU-only torch to avoid pulling multi-GB CUDA dependencies
$ENV/bin/python -m pip install torch --index-url https://download.pytorch.org/whl/cpu
$ENV/bin/python -m pip install vbll==0.4.9 pandas numpy scikit-learn matplotlib

# Install and register the Jupyter kernel — avoid --user (writes to nashome, will fail on quota)
mkdir -p /exp/icarus/data/users/sdey2/jupyter
$ENV/bin/python -m pip install --no-cache-dir --only-binary :all: ipykernel
$ENV/bin/python -m ipykernel install --prefix=/exp/icarus/data/users/sdey2/jupyter \
    --name vbll_repro --display-name "VBLL Repro (py3.12)"

# Make the kernel discoverable — add to ~/.bashrc to persist
export JUPYTER_PATH=/exp/icarus/data/users/sdey2/jupyter/share/jupyter:$JUPYTER_PATH
```

### Potential errors you may come across 
- **`conda activate` unreliable in notebook subshells** — each `!` cell is a fresh subshell without conda's init hooks loaded. Call binaries by full path instead: `$ENV/bin/python`, not `conda activate vbll_repro`.
- **`pip` script itself can have a broken/stale shebang** if the env was ever moved or partially rebuilt — if `$ENV/bin/pip ...` fails with "bad interpreter," use `$ENV/bin/python -m pip ...` instead, which bypasses the shebang entirely.
- **Packages requiring C compilation** (e.g. `pyzmq`) may fail on this node's older system `gcc` (pre-C99 mode) — prefer prebuilt wheels: `pip install --only-binary :all: <package>`.
- **Corrupted pip cache** can cause a repeating `IncompleteRead` error on the exact same byte count — fix with `pip install --no-cache-dir ...`.
- **`jupyter <subcommand>` (e.g. `jupyter kernelspec list`) dispatches via `PATH` lookup, not the invoking interpreter** — always call `$ENV/bin/jupyter ...` directly, not `$ENV/bin/python -m jupyter ...`, to avoid accidentally invoking a different, stale `jupyter` elsewhere on `PATH`.

In [2]:
import sys
print(sys.executable)

import torch, vbll
import subprocess
print(subprocess.run([sys.executable, '-m', 'pip', 'show', 'vbll'], capture_output=True, text=True).stdout)

/exp/icarus/data/users/sdey2/vbll_surrogate/conda/envs/vbll_repro/bin/python
Name: vbll
Version: 0.4.9
Summary: 
Home-page: 
Author: John Willes
Author-email: johnwilles@gmail.com
License: 
Location: /exp/icarus/data/users/sdey2/vbll_surrogate/conda/envs/vbll_repro/lib/python3.12/site-packages
Requires: numpy, torch
Required-by: 



In [3]:
import os
os.chdir('/exp/icarus/app/users/sdey2/FDPSurrogateModel/VBLL_SurrogateModel/code')
print(os.getcwd())
print(os.listdir('.'))   # should show data.py, model.py, train.py, run.py, run_het.py, etc.

/exp/icarus/app/users/sdey2/FDPSurrogateModel/VBLL_SurrogateModel/code
['plotting.py', 'train.py', 'run.py', 'data.py', 'vbll_patches.py', 'run_het.py', 'evaluate.py', 'model.py', '__pycache__', 'analyze_uncertainty.py']


In [4]:
CHECKPOINT = '../output_data/checkpoints/best_model_v0_std.pt'
print(os.path.exists(CHECKPOINT))   # confirm True before proceeding

print(repr(CHECKPOINT))   # see the exact value being used
print(os.getcwd())        # confirm you're still in code/
print(os.path.exists(CHECKPOINT))

True
'../output_data/checkpoints/best_model_v0_std.pt'
/exp/icarus/app/users/sdey2/FDPSurrogateModel/VBLL_SurrogateModel/code
True


After you run the code once, i.e. `run.py` and `run_het.py`, you can replay and extract the summary statistics again by using the saved checkpoints in 
`/exp/icarus/app/users/sdey2/FDPSurrogateModel/VBLL_SurrogateModel/output_data/checkpoints` rather than having to re-run the VBLL algorithm every time. 

We do this in the cell below. Make sure that the checkpoint, input data file location and the "head type" (i.e None or het) are correct! Mismatches may give you errors!

In [13]:
import torch
from data import build_loaders
from model import ParticleSurrogate
import evaluate as ev

#CHECKPOINT = 'best_model_v0_std.pt'   # adjust per checkpoint
#DATA_PATH  = '/exp/icarus/app/users/sdey2/FDPSurrogateModel/VBLL_SurrogateModel/input_data/v0/masteranadev_selected_events.csv'
#HEAD_TYPE  = None   # or 'het' — must match how this checkpoint was trained

CHECKPOINT = '../output_data/checkpoints/best_model_v0_het.pt'
DATA_PATH  = '../input_data/v0/masteranadev_selected_events.csv'
#DATA_PATH  = '/exp/icarus/app/users/sdey2/FDPSurrogateModel/VBLL_SurrogateModel/input_data/x60/MINERvALargeDataset_x60_v1/masteranadev_selected_events_x60.csv'
HEAD_TYPE  = 'het'

train_loader, val_loader, outlier_loader, normaliser, n_train_per_particle = \
    build_loaders(DATA_PATH, batch_size=64)

kwargs = dict(
    n_train_per_particle=n_train_per_particle,
    d_embed=8, hidden=64, n_layers=3,
    wishart_scale=1.0, prior_scale=1.0, dof=1.0,
)
if HEAD_TYPE == 'het':
    kwargs.update(head_type='het', noise_prior_scale=0.01)



print(os.path.exists(CHECKPOINT))   # confirm True before proceeding

model = ParticleSurrogate(**kwargs)
model.load_state_dict(torch.load(CHECKPOINT))
model.eval()

val_preds     = ev.collect_predictions(model, val_loader)
outlier_preds = ev.collect_predictions(model, outlier_loader)

print("\n-- Pull statistics --");            ev.pull_statistics(val_preds)
print("\n-- Sigma variation (CV) --");        ev.sigma_variation(val_preds)
print("\n-- Coverage --");                    ev.coverage(val_preds)
print("\n-- Uncertainty vs residual --");     ev.uncertainty_vs_residual(val_preds)
print("\n-- Outlier uncertainty probe --");   ev.outlier_uncertainty_probe(model, val_loader, outlier_loader)

Loading single input file: ../input_data/v0/masteranadev_selected_events.csv
Parsed 3,337 events
Flagged 59 outlier events (1.8%)
Random train/val split: 1,639 train / 1,639 val clean events (train_fraction=0.50, seed=42)
True

-- Pull statistics --

  muon  (n=1639):
    E  : mean=-0.082,  std=1.153  (ideal: 0, 1)
    px : mean=+0.105,  std=0.906  (ideal: 0, 1)
    py : mean=+0.034,  std=0.962  (ideal: 0, 1)
    pz : mean=-0.091,  std=1.146  (ideal: 0, 1)

  proton  (n=1639):
    E  : mean=-0.008,  std=1.025  (ideal: 0, 1)
    px : mean=-0.097,  std=0.910  (ideal: 0, 1)
    py : mean=+0.162,  std=0.993  (ideal: 0, 1)
    pz : mean=+0.013,  std=1.128  (ideal: 0, 1)

-- Sigma variation (CV) --

  muon:
    E  : mean_σ=0.5383,  std_σ=0.2035,  CV=0.3781  ✓
    px : mean_σ=0.3579,  std_σ=0.2150,  CV=0.6009  ✓
    py : mean_σ=0.3525,  std_σ=0.2068,  CV=0.5868  ✓
    pz : mean_σ=0.5357,  std_σ=0.2021,  CV=0.3773  ✓

  proton:
    E  : mean_σ=0.7907,  std_σ=0.1744,  CV=0.2205  ✓
    px : mean